# 7. Preprocessing and Feature Extraction

This notebook presents the main preprocessing and feature extraction tools available in the 3W Toolkit.

The goal is to understand the typical workflow:

**prepare the signals → handle missing data → prepare windows → extract informative features**

We use real 3W events throughout the examples.

## Learning Goals

By the end of this notebook, you will be able to:

- Understand the preprocessing and feature extraction workflow for time-series data.
- Compose preprocessing steps using sequential adapters.
- Clean signals and handle missing values using the available preprocessing tools.
- Segment time-series data into windows using configurable windowing strategies.
- Extract statistical, wavelet, and exponentially weighted statistical features.
- Combine multiple feature extraction strategies into a single feature representation.
- Build reusable preprocessing and feature extraction pipelines with the 3W Toolkit.


## 📋 Table of Contents

1. [Data preparation](#data-preparation)
    1. [Loading the 3W Dataset](#load-the-3w-dataset)
2. [Adapters for composing processing steps](#adapters-processing)
3. [Cleaning signals](#cleaning)
4. [Handling missing values](#handling-missings)
5. [Windowing](#windowing)
6. [Statistical feature extraction](#stats-feat-extraction)
7. [Wavelet feature extraction](#wavelet-feat)
8. [Exponentially weighted statistical features](#exp-weight-stats-feat)
9. [Combining feature extractors](#combining-feat)
10. [Next Steps](#next-steps)
---

## 1. Data preparation <a id="data-preparation"></a>

Preprocessing utilities are configured as transformation steps and can be composed into sequential pipelines.

Feature extractors operate on **pre-windowed data**, so the usual workflow is:

1. preprocess the signals;
2. divide them into windows;
3. extract features from each window.


**Packages**

In [1]:
from ThreeWToolkit.preprocessing import (
    CleanSignalsConfig,
    ImputeMissingConfig,
    NormalizeConfig,
    SequentialPreprocessingAdapterConfig,
)

from ThreeWToolkit.feature_extraction import (
    WindowingConfig,
    StatisticalConfig,
    WaveletConfig,
    EWStatisticalConfig,
    SequentialFeatureAdapterConfig,
    ConcatFeatureAdapterConfig,
)

from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig


### Load the 3W dataset <a id="load-the-3w-dataset"></a>

In [2]:
dataset_path = "../../dataset"

ds_real = ParquetDatasetConfig(
    path=dataset_path,
    event_type=["real"],
).build()

print(f"Number of events: {len(ds_real)}")


2026-09-28 16:26:48,753 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset found at ..\..\dataset
2026-09-28 16:26:48,758 | INFO | ThreeWToolkit.dataset.parquet_dataset | Validating dataset integrity...
2026-09-28 16:26:48,759 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset integrity check passed!


Number of events: 1119


filter data instances

In [3]:
ds = [ds_real[i] for i in range(100)]

## 2. Adapters for composing processing steps <a id="adapters-processing"></a>

Adapters are used to compose and organize preprocessing and feature extraction steps into reusable pipelines.

### Sequential adapter

`SequentialPreprocessingAdapterConfig` applies preprocessing steps **one after another**, in the order they are defined.

This is useful when one preprocessing operation should be followed by another. For example, we can clean the signals first and then handle missing values:

In [4]:
preprocessing_pipeline = SequentialPreprocessingAdapterConfig(
    steps=[
        CleanSignalsConfig(),
        ImputeMissingConfig(strategy="mean"),
    ]
)

The output of each step becomes the input of the next one.

### Concat adapter

`ConcatFeatureAdapterConfig` combines the outputs of multiple feature extractors into a single feature representation.

For example, statistical, exponentially weighted statistical, and wavelet features can be extracted together:

In [5]:
feature_pipeline = ConcatFeatureAdapterConfig(
    steps=[
        StatisticalConfig(features=["mean", "std", "min", "max"]),
        EWStatisticalConfig(window_size=128, decay=0.9),
        WaveletConfig(level=7)
    ]
)

This allows different feature extraction strategies to be combined into one feature set.

## 3. Cleaning signals <a id="cleaning"></a>

`CleanSignalsConfig` identifies frozen or out-of-range behavior using thresholds based on the interquartile range (IQR) of signal statistics.

Detected faulty values are replaced with `NaN`, and columns with too many missing values can be removed according to the configured threshold.

This IQR-based cleaning is intended for continuous signals.


In [6]:
clean_transformer = TransformConfig(
    pre_processing=CleanSignalsConfig()
).build()

clean_transformer.fit(ds)
clean_ds = clean_transformer.transform(ds)

print(f"Columns before cleaning: {ds[0].signal.shape[1]}")
print(f"Columns after cleaning:  {clean_ds[0].signal.shape[1]}")


Columns before cleaning: 28
Columns after cleaning:  16


## 4. Handling missing values <a id="handling-missings"></a>

`ImputeMissingConfig` supports several strategies:

- `mean`
- `median`
- `constant`
- `ffill`
- `bfill`
- `interpolate`

The appropriate choice depends on the signal and on what the missing values represent.


In [7]:
preprocessing_pipeline = SequentialPreprocessingAdapterConfig(
    steps=[
        CleanSignalsConfig(),
        ImputeMissingConfig(strategy="mean"),
    ]
)

preprocessor = TransformConfig(
    pre_processing=preprocessing_pipeline
).build()

preprocessor.fit(ds)
preprocessed_ds = preprocessor.transform(ds)

preprocessed_ds[0].signal.head()


,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,ESTADO-XO,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-TPT,T-JUS-CKP,T-TPT,state
timestamp,,,,,,,,,,,,,,,,
2017-02-01 01:02:07,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,1563422.0,1627884.0,10074540.0,84.64463,119.0781,0.0
2017-02-01 01:02:08,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,1563422.0,1633397.0,10074540.0,84.63828,119.0781,0.0
2017-02-01 01:02:09,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,1563422.0,1638909.0,10074540.0,84.63194,119.0781,0.0
2017-02-01 01:02:10,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,1563422.0,1644422.0,10074540.0,84.62558,119.0781,0.0
2017-02-01 01:02:11,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,1563423.0,1649934.0,10074540.0,84.61923,119.0781,0.0


### Other preprocessing utilities

The toolkit also provides `NormalizeConfig`, `RenameColumnsConfig`, `FillLabelsConfig`, and `RemapClassConfig`.

They can be added when required by the dataset or downstream task. For example, normalization supports L1, L2, and max norms.


In [8]:
normalization_pipeline = SequentialPreprocessingAdapterConfig(
    steps=[
        CleanSignalsConfig(),
        ImputeMissingConfig(strategy="mean"),
        NormalizeConfig(norm="l2"),
    ]
)

normalizer = TransformConfig(
    pre_processing=normalization_pipeline
).build()

normalizer.fit(ds)
normalized_ds = normalizer.transform(ds)

normalized_ds[0].signal.head()


,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,ESTADO-XO,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-TPT,T-JUS-CKP,T-TPT,state
timestamp,,,,,,,,,,,,,,,,
2017-02-01 01:02:07,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.68905,-0.615386,-1.321588,-1.200091,1.552551,1.890326,0.0
2017-02-01 01:02:08,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.68905,-0.615386,-1.319012,-1.200091,1.551596,1.890326,0.0
2017-02-01 01:02:09,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.68905,-0.615386,-1.316437,-1.200091,1.550642,1.890326,0.0
2017-02-01 01:02:10,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.68905,-0.615386,-1.313861,-1.200091,1.549685,1.890326,0.0
2017-02-01 01:02:11,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.68905,-0.615385,-1.311286,-1.200091,1.548729,1.890326,0.0


## 5. Windowing <a id="windowing"></a>

`WindowingConfig` segments a time series into windows.

Important parameters include:

- `window_size`: samples per window;
- `overlap`: fraction shared by consecutive windows;
- `window`: window function such as `"hann"` or `"hamming"`;
- `pad_last_window`: whether to pad an incomplete final window.

Window size and overlap should be selected according to the sampling rate and temporal patterns of interest.


In [9]:
window_transformer = TransformConfig(
    pre_processing=preprocessing_pipeline,
    feature_extraction=WindowingConfig(
        window_size=128,
        overlap=0.0,
    ),
).build()

window_transformer.fit(ds)
windowed_ds = window_transformer.transform(ds)

print(f"Windowed signal shape: {windowed_ds[0].signal.shape}")


Windowed signal shape: (2688, 128)


## 6. Statistical feature extraction <a id="stats-feat-extraction"></a>

`StatisticalConfig` summarizes each window using statistical descriptors.

Common descriptors include `mean`, `std`, `min`, `max`, `skew`, `kurtosis`, and quartiles.

For the workshop, we use four basic descriptors to keep the example focused.


In [10]:
stat_transformer = TransformConfig(
    pre_processing=preprocessing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=128),
            StatisticalConfig(features=["mean", "std", "min", "max"]),
        ]
    ),
).build()

stat_transformer.fit(ds)
statistical_ds = stat_transformer.transform(ds)

print(f"Feature matrix shape: {statistical_ds[0].signal.shape}")
statistical_ds[0].signal.head()


Feature matrix shape: (168, 64)


,mean_ESTADO-DHSV,mean_ESTADO-M1,mean_ESTADO-M2,mean_ESTADO-PXO,mean_ESTADO-SDV-GL,mean_ESTADO-SDV-P,mean_ESTADO-W1,mean_ESTADO-W2,mean_ESTADO-XO,mean_P-ANULAR,...,max_ESTADO-W1,max_ESTADO-W2,max_ESTADO-XO,max_P-ANULAR,max_P-JUS-CKGL,max_P-MON-CKP,max_P-TPT,max_T-JUS-CKP,max_T-TPT,max_state
window,,,,,,,,,,,,,,,,,,,,,
0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563422.0,1627884.0,10074540.0,84.64463,119.0781,0.0
1,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563448.0,1812740.0,10074540.0,85.05927,119.0786,0.0
2,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563473.0,1947285.0,10074540.0,85.30231,119.0792,0.0
3,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563499.0,2021068.0,10074540.0,84.68998,119.0798,0.0
4,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563525.0,1965369.0,10074540.0,84.54279,119.0804,0.0


## 7. Wavelet feature extraction <a id="wavelet-feat"></a>

`WaveletConfig` uses the Stationary Wavelet Transform (SWT) to capture information at different frequency scales.

- **Approximation coefficients (A)** capture lower-frequency structure and trends.
- **Detail coefficients (D)** capture higher-frequency behavior such as abrupt changes and spikes.

Wavelet features complement statistical summaries when multi-scale signal structure matters.


In [11]:
wavelet_transformer = TransformConfig(
    pre_processing=preprocessing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(
                window_size=128,
                overlap=0.875,
                window="boxcar",
            ),
            WaveletConfig(level=7),
        ]
    ),
).build()

wavelet_transformer.fit(ds)
wavelet_ds = wavelet_transformer.transform(ds)

print(f"Wavelet feature shape: {wavelet_ds[0].signal.shape}")
wavelet_ds[0].signal.head()


Wavelet feature shape: (1343, 144)


,A7_ESTADO-DHSV,A7_ESTADO-M1,A7_ESTADO-M2,A7_ESTADO-PXO,A7_ESTADO-SDV-GL,A7_ESTADO-SDV-P,A7_ESTADO-W1,A7_ESTADO-W2,A7_ESTADO-XO,A7_P-ANULAR,...,A0_ESTADO-W1,A0_ESTADO-W2,A0_ESTADO-XO,A0_P-ANULAR,A0_P-JUS-CKGL,A0_P-MON-CKP,A0_P-TPT,A0_T-JUS-CKP,A0_T-TPT,A0_state
window,,,,,,,,,,,,,,,,,,,,,
0,11.313708,11.313708,0.0,0.0,0.0,11.313708,11.313708,0.0,0.0,1.444504e+08,...,1.0,0.0,0.0,12767730.0,1563422.0,1627884.0,10074540.0,84.64463,119.0781,0.0
1,11.313708,11.313708,0.0,0.0,0.0,11.313708,11.313708,0.0,0.0,1.444504e+08,...,1.0,0.0,0.0,12767730.0,1563425.0,1716084.0,10074540.0,84.54303,119.0781,0.0
2,11.313708,11.313708,0.0,0.0,0.0,11.313708,11.313708,0.0,0.0,1.444504e+08,...,1.0,0.0,0.0,12767730.0,1563428.0,1622497.0,10074540.0,84.50632,119.0782,0.0
3,11.313708,11.313708,0.0,0.0,0.0,11.313708,11.313708,0.0,0.0,1.444504e+08,...,1.0,0.0,0.0,12767730.0,1563431.0,1746001.0,10074540.0,84.66431,119.0783,0.0
4,11.313708,11.313708,0.0,0.0,0.0,11.313708,11.313708,0.0,0.0,1.444504e+08,...,1.0,0.0,0.0,12767730.0,1563435.0,1807315.0,10074540.0,84.82229,119.0783,0.0


## 8. Exponentially weighted statistical features <a id="exp-weight-stats-feat"></a>

`EWStatisticalConfig` gives more weight to recent observations within a window.

The `decay` parameter controls how quickly the influence of older observations decreases.

This is useful when recent signal behavior is more relevant than older observations in the same window.


In [12]:
ew_transformer = TransformConfig(
    pre_processing=preprocessing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=128),
            EWStatisticalConfig(window_size=128, decay=0.9),
        ]
    ),
).build()

ew_transformer.fit(ds)
ew_ds = ew_transformer.transform(ds)

print(f"EW feature shape: {ew_ds[0].signal.shape}")
ew_ds[0].signal.head()


EW feature shape: (168, 144)


,ew_mean_ESTADO-DHSV,ew_mean_ESTADO-M1,ew_mean_ESTADO-M2,ew_mean_ESTADO-PXO,ew_mean_ESTADO-SDV-GL,ew_mean_ESTADO-SDV-P,ew_mean_ESTADO-W1,ew_mean_ESTADO-W2,ew_mean_ESTADO-XO,ew_mean_P-ANULAR,...,ew_max_ESTADO-W1,ew_max_ESTADO-W2,ew_max_ESTADO-XO,ew_max_P-ANULAR,ew_max_P-JUS-CKGL,ew_max_P-MON-CKP,ew_max_P-TPT,ew_max_T-JUS-CKP,ew_max_T-TPT,ew_max_state
window,,,,,,,,,,,,,,,,,,,,,
0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.157018,0.000000,0.000000,0.0
1,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,0.0,0.0,0.0,0.0,1.131275,2.507936,0.157018,7.124940,0.351390,0.0
2,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,0.0,0.0,0.0,0.0,0.761720,2.218195,0.157018,2.707456,0.480122,0.0
3,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,0.0,0.0,0.0,0.0,0.895749,2.082705,0.157018,22.349038,0.588199,0.0
4,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,0.0,0.0,0.0,0.0,0.941564,8.840530,0.157018,3.914630,0.798235,0.0


## 9. Combining feature extractors <a id="combining-feat"></a>

`ConcatFeatureAdapterConfig` can concatenate several feature extraction outputs.

A richer pipeline can therefore combine statistical, exponentially weighted, and wavelet features after a single windowing step.


In [13]:
combined = TransformConfig(
    pre_processing=preprocessing_pipeline,
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=128),
            ConcatFeatureAdapterConfig(
                steps=[
                    StatisticalConfig(features=["mean", "std", "min", "max"]),
                    EWStatisticalConfig(window_size=128, decay=0.9),
                    WaveletConfig(level=7),
                ]
            ),
        ]
    ),
).build()

combined.fit(ds)
features_ds = combined.transform(ds)

event_features = features_ds[0].signal
print(f"Final feature matrix shape: {event_features.shape}")
event_features.head()


Final feature matrix shape: (168, 352)


,mean_ESTADO-DHSV,mean_ESTADO-M1,mean_ESTADO-M2,mean_ESTADO-PXO,mean_ESTADO-SDV-GL,mean_ESTADO-SDV-P,mean_ESTADO-W1,mean_ESTADO-W2,mean_ESTADO-XO,mean_P-ANULAR,...,A0_ESTADO-W1,A0_ESTADO-W2,A0_ESTADO-XO,A0_P-ANULAR,A0_P-JUS-CKGL,A0_P-MON-CKP,A0_P-TPT,A0_T-JUS-CKP,A0_T-TPT,A0_state
window,,,,,,,,,,,,,,,,,,,,,
0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563422.0,1627884.0,10074540.0,84.64463,119.0781,0.0
1,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563448.0,1647814.0,10074540.0,84.07881,119.0786,0.0
2,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563473.0,1386319.0,10074540.0,84.71109,119.0792,0.0
3,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563499.0,1353551.0,10074540.0,84.44953,119.0798,0.0
4,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,12767730.0,...,1.0,0.0,0.0,12767730.0,1563525.0,1568442.0,10074540.0,84.38683,119.0804,0.0


## Next steps <a id="next-steps"></a>

🎉 **Nice!** Now you can use the **3W Toolkit** preprocessing and feature extraction tools!

These tools demonstrates the capabilities of the toolkit focusing on preprocessing and transforming raw time-series data into meaningful features for machine learning applications.

The feature extraction tools in this package operate at event-level.

### What's Next?

1. **Model Training**: Discover how to train machine learning models in [Notebook 8: Model Training](8_model_training.ipynb)
2. **Model Evaluating and Metrics**: Discover how to evaluate machine learning models and use available metrics in [Notebook 9: Model Evaluation and Metrics](9_metrics_and_evaluation.ipynb)
---